# Chess Tutor — Prompt Strategy Comparison
**STA561D: Probabilistic Machine Learning**

Controlled experiment: four prompt engineering strategies evaluated across four ELO bands.
Position and move held constant. Treatment variable: prompt design. Outcome variables: readability and vocabulary complexity metrics.

| Strategy | Description |
|----------|-------------|
| **Minimal** | ELO stated, explanation requested. No structure, no constraints. Baseline. |
| **Role-play** | Persona framing — model told to act as a coach for a specific player type. No explicit rules. |
| **Chain-of-thought** | Two-step: model first reasons about player's knowledge level, then writes explanation. |
| **Structured** | Current system — three mandatory sections, explicit forbidden vocabulary list, depth instructions. |

## Setup

In [ ]:
import os
import sys
import asyncio
import chess
import chess.svg
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import textstat
import anthropic
from IPython.display import SVG, display
from dotenv import load_dotenv

load_dotenv()

if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))

print("Setup complete.")

## Test Position and Move (same as demo notebook)

In [ ]:
TEST_FEN = "r1bqk2r/pppp1ppp/2n2n2/2b1p3/2B1P3/2N2N2/PPPP1PPP/R1BQK2R w KQkq - 4 5"
MOVE_SAN = "O-O"       # Kingside castle — rich in strategic content across all ELO levels
MOVE_UCI = "e1g1"
EVALUATION = "+0.19"
ALTERNATIVES = ["Bxf7+", "Be6"]

board = chess.Board(TEST_FEN)
svg = chess.svg.board(board, size=300)
display(SVG(svg))
print(f"Position: Ruy Lopez mid-game | Move: {MOVE_SAN} | Eval: {EVALUATION}")
print(f"Alternatives for comparison: {ALTERNATIVES}")

## ELO Profiles for Prompt Construction

In [ ]:
ELO_LEVELS = [800, 1200, 1600, 1800]

ELO_META = {
    800:  {
        "label": "Beginner (800)",
        "persona": "a complete beginner who just learned how the pieces move and has been playing for a few weeks",
        "forbidden": ["tempo", "initiative", "outpost", "prophylaxis", "compensation",
                      "zwischenzug", "battery", "imbalance", "dynamic", "pawn structure",
                      "piece coordination", "open file", "weak square"],
        "depth_instruction": "Think one move ahead only.",
        "cot_question": "What does a complete beginner (ELO ~800) who just learned the rules understand about chess?",
    },
    1200: {
        "label": "Intermediate (1200)",
        "persona": "a casual club player who knows basic tactics like forks and pins but struggles with long-term strategy",
        "forbidden": ["tempo", "initiative", "outpost", "prophylaxis", "compensation",
                      "zwischenzug", "battery", "dynamic imbalance", "zugzwang"],
        "depth_instruction": "Think 2-3 moves ahead.",
        "cot_question": "What does a casual chess player (ELO ~1200) who knows basic tactics understand about chess strategy?",
    },
    1600: {
        "label": "Club Player (1600)",
        "persona": "a strong club player who understands piece activity, pawn structure, and common tactical patterns",
        "forbidden": [],
        "depth_instruction": "Describe a 4-5 move plan.",
        "cot_question": "What does a strong club player (ELO ~1600) who understands positional chess know about this type of position?",
    },
    1800: {
        "label": "Advanced (1800)",
        "persona": "an advanced player with strong tactical vision and understanding of strategic imbalances",
        "forbidden": [],
        "depth_instruction": "Discuss long-term structural implications and opponent counterplay.",
        "cot_question": "What does an advanced player (ELO ~1800) with strong pattern recognition understand about this type of middlegame?",
    },
}

## Prompt Builders — Four Strategies

In [ ]:
def prompt_minimal(elo):
    """Strategy 1: Minimal — ELO stated, no structure, no constraints."""
    return f"""You are a chess tutor. The player is rated ELO {elo}.

Position FEN: {TEST_FEN}
Recommended move: {MOVE_SAN}
Position evaluation: {EVALUATION}

Explain why {MOVE_SAN} is a good move for this player. Keep it under 200 words."""


def prompt_roleplay(elo):
    """Strategy 2: Role-play — persona framing, no explicit rules."""
    meta = ELO_META[elo]
    return f"""You are an experienced chess coach speaking directly to {meta['persona']}.
Adapt your language, vocabulary, and depth of explanation entirely to suit this player.
Do not use jargon they would not understand.

Position FEN: {TEST_FEN}
You are recommending the move: {MOVE_SAN}
Position evaluation: {EVALUATION}
Alternative moves they might consider: {', '.join(ALTERNATIVES)}

Explain: why {MOVE_SAN} is the right move, what they should do next, and why the alternatives are weaker.
Keep it under 200 words. Speak directly to the player."""


def prompt_cot(elo):
    """Strategy 3: Chain-of-thought — model reasons about player first, then writes explanation."""
    meta = ELO_META[elo]
    return f"""You are a chess tutor. Before writing your explanation, reason through what the player understands.

Step 1 — Reason (do not show this to the player): {meta['cot_question']}
List what vocabulary they know, what concepts they grasp, and how far ahead they can think.

Step 2 — Write the explanation for the player based on your reasoning in Step 1.
Position FEN: {TEST_FEN}
Recommended move: {MOVE_SAN}
Evaluation: {EVALUATION}
Alternative moves: {', '.join(ALTERNATIVES)}

Format your response as:
REASONING: [your internal reasoning about the player]
EXPLANATION: [the actual explanation for the player, under 200 words]"""


def prompt_structured(elo):
    """Strategy 4: Structured — current system with mandatory sections and forbidden vocabulary."""
    meta = ELO_META[elo]
    forbidden_str = (
        f"Never use these terms: {', '.join(meta['forbidden'])}."
        if meta['forbidden'] else "Full technical vocabulary is appropriate."
    )
    return f"""You are a chess tutor explaining a move to a player rated ELO {elo}.

Position FEN: {TEST_FEN}
Recommended move: {MOVE_SAN}
Evaluation: {EVALUATION}
Alternative moves: {', '.join(ALTERNATIVES)}

YOUR RESPONSE MUST HAVE EXACTLY THREE SECTIONS:

**Why this move?**
Explain why {MOVE_SAN} is the best choice.
{forbidden_str}
{meta['depth_instruction']}

**Your strategy going forward**
Tell the player what to do over the next few moves. {meta['depth_instruction']}

**How does this compare to other moves?**
Compare against: {', '.join(ALTERNATIVES)}. What goes wrong with each?

HARD RULES: Total 150-200 words. Do not use vocabulary forbidden above. All three sections required."""


STRATEGIES = {
    "Minimal":    prompt_minimal,
    "Role-play":  prompt_roleplay,
    "Chain-of-thought": prompt_cot,
    "Structured": prompt_structured,
}

print(f"Strategies defined: {list(STRATEGIES.keys())}")
print(f"ELO levels: {ELO_LEVELS}")
print(f"Total API calls: {len(STRATEGIES) * len(ELO_LEVELS)}")

## Generate All Explanations (16 API calls)

In [ ]:
def call_api(prompt):
    message = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=500,
        messages=[{"role": "user", "content": prompt}]
    )
    return message.content[0].text.strip()


def extract_explanation(text, strategy):
    """For CoT strategy, extract only the EXPLANATION section for fair metric comparison."""
    if strategy == "Chain-of-thought" and "EXPLANATION:" in text:
        parts = text.split("EXPLANATION:")
        return parts[1].strip() if len(parts) > 1 else text
    return text


# Run all 16 calls
results = {}

for strategy_name, prompt_fn in STRATEGIES.items():
    results[strategy_name] = {}
    for elo in ELO_LEVELS:
        print(f"  {strategy_name} | ELO {elo}...", end=" ", flush=True)
        prompt = prompt_fn(elo)
        raw = call_api(prompt)
        explanation = extract_explanation(raw, strategy_name)
        results[strategy_name][elo] = {
            "raw": raw,
            "explanation": explanation
        }
        print(f"done ({len(explanation.split())} words)")

print("\nAll 16 calls complete.")

## Compute Metrics

In [ ]:
ADVANCED_TERMS = [
    'tempo', 'initiative', 'outpost', 'prophylaxis', 'compensation',
    'zwischenzug', 'battery', 'fianchetto', 'pawn structure', 'imbalance',
    'open file', 'weak square', 'piece coordination', 'dynamic', 'static',
    'zugzwang', 'overloaded', 'deflection', 'interference', 'decoy'
]

rows = []
for strategy_name in STRATEGIES:
    for elo in ELO_LEVELS:
        text = results[strategy_name][elo]["explanation"]
        text_lower = text.lower()
        found_terms = [t for t in ADVANCED_TERMS if t in text_lower]

        rows.append({
            "Strategy":            strategy_name,
            "ELO":                 elo,
            "ELO Label":           ELO_META[elo]["label"],
            "FK Grade Level":      round(textstat.flesch_kincaid_grade(text), 2),
            "Flesch Reading Ease": round(textstat.flesch_reading_ease(text), 2),
            "Avg Syllables/Word":  round(textstat.avg_syllables_per_word(text), 2),
            "Advanced Terms Used": len(found_terms),
            "Terms Found":         ", ".join(found_terms) if found_terms else "none",
            "Word Count":          len(text.split()),
        })

df = pd.DataFrame(rows)
print(df[['Strategy','ELO Label','FK Grade Level','Flesch Reading Ease','Avg Syllables/Word','Advanced Terms Used']].to_string(index=False))

## Chart 1: FK Grade Level Across ELO by Strategy
Key question: which strategy shows the most consistent monotonic increase in reading difficulty as ELO rises?

In [ ]:
strategy_colors = {
    "Minimal":           "#95a5a6",
    "Role-play":         "#3498db",
    "Chain-of-thought":  "#e67e22",
    "Structured":        "#2ecc71",
}

fig, ax = plt.subplots(figsize=(10, 6))

for strategy_name in STRATEGIES:
    subset = df[df['Strategy'] == strategy_name].sort_values('ELO')
    ax.plot(
        subset['ELO'],
        subset['FK Grade Level'],
        marker='o', linewidth=2.5, markersize=8,
        label=strategy_name,
        color=strategy_colors[strategy_name]
    )
    # Annotate endpoints
    for _, row in subset.iterrows():
        ax.annotate(
            f"{row['FK Grade Level']:.1f}",
            (row['ELO'], row['FK Grade Level']),
            textcoords="offset points", xytext=(0, 10),
            ha='center', fontsize=8, color=strategy_colors[strategy_name]
        )

ax.set_xlabel('Player ELO', fontsize=12)
ax.set_ylabel('Flesch-Kincaid Grade Level', fontsize=12)
ax.set_title('Reading Complexity by Prompt Strategy and ELO\n(Higher = more complex language)', fontsize=13, fontweight='bold')
ax.set_xticks(ELO_LEVELS)
ax.set_xticklabels([str(e) for e in ELO_LEVELS])
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 16)

plt.tight_layout()
plt.savefig('strategy_comparison_fk.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: strategy_comparison_fk.png")

## Chart 2: Advanced Vocabulary Usage by Strategy and ELO
Key question: which strategy best suppresses technical terms at low ELO?

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 5), sharey=True)
fig.suptitle('Advanced Vocabulary Usage per ELO Band\n(Lower is better at low ELO)',
             fontsize=13, fontweight='bold')

for i, elo in enumerate(ELO_LEVELS):
    ax = axes[i]
    subset = df[df['ELO'] == elo]
    strategies = subset['Strategy'].tolist()
    counts = subset['Advanced Terms Used'].tolist()
    colors = [strategy_colors[s] for s in strategies]

    bars = ax.bar(strategies, counts, color=colors, width=0.6)
    ax.set_title(f'ELO {elo}', fontweight='bold')
    ax.set_xticks(range(len(strategies)))
    ax.set_xticklabels(strategies, rotation=30, ha='right', fontsize=9)
    ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

    for bar, val in zip(bars, counts):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                    str(val), ha='center', va='bottom', fontweight='bold')

axes[0].set_ylabel('Advanced Terms Used')
plt.tight_layout()
plt.savefig('strategy_comparison_vocab.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: strategy_comparison_vocab.png")

## Chart 3: Monotonicity Score
Measures how consistently each strategy scales complexity with ELO. A perfectly calibrated strategy should show monotonically increasing FK Grade Level. We compute the fraction of consecutive ELO pairs where complexity increases.

In [ ]:
def monotonicity_score(strategy_name):
    """Fraction of consecutive ELO pairs where FK Grade Level increases."""
    subset = df[df['Strategy'] == strategy_name].sort_values('ELO')
    grades = subset['FK Grade Level'].tolist()
    pairs = [(grades[i], grades[i+1]) for i in range(len(grades)-1)]
    increasing = sum(1 for a, b in pairs if b > a)
    return round(increasing / len(pairs), 2)

def fk_range(strategy_name):
    """Range of FK Grade Level from lowest to highest ELO."""
    subset = df[df['Strategy'] == strategy_name].sort_values('ELO')
    grades = subset['FK Grade Level'].tolist()
    return round(grades[-1] - grades[0], 2)

def vocab_compliance(strategy_name):
    """Average advanced terms used at ELO 800 and 1000 — lower is better."""
    low_elo = df[(df['Strategy'] == strategy_name) & (df['ELO'] <= 1200)]
    return round(low_elo['Advanced Terms Used'].mean(), 2)

summary_rows = []
for s in STRATEGIES:
    summary_rows.append({
        "Strategy":              s,
        "Monotonicity Score":    monotonicity_score(s),
        "FK Range (800→1800)":   fk_range(s),
        "Avg Terms at Low ELO":  vocab_compliance(s),
    })

df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Prompt Strategy Comparison — Calibration Quality Metrics',
             fontsize=13, fontweight='bold')

strat_names = df_summary['Strategy'].tolist()
colors = [strategy_colors[s] for s in strat_names]

# Plot 1: Monotonicity
bars1 = axes[0].bar(strat_names, df_summary['Monotonicity Score'], color=colors)
axes[0].set_title('Monotonicity Score\n(1.0 = perfectly consistent scale-up)', fontweight='bold')
axes[0].set_ylabel('Score (0 to 1.0)')
axes[0].set_ylim(0, 1.2)
axes[0].set_xticks(range(len(strat_names)))
axes[0].set_xticklabels(strat_names, rotation=15, ha='right')
for bar, val in zip(bars1, df_summary['Monotonicity Score']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{val:.2f}', ha='center', fontweight='bold')

# Plot 2: FK Range
bars2 = axes[1].bar(strat_names, df_summary['FK Range (800→1800)'], color=colors)
axes[1].set_title('FK Grade Level Range\n(Higher = wider calibration spread)', fontweight='bold')
axes[1].set_ylabel('Grade Levels')
axes[1].set_xticks(range(len(strat_names)))
axes[1].set_xticklabels(strat_names, rotation=15, ha='right')
for bar, val in zip(bars2, df_summary['FK Range (800→1800)']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{val:.1f}', ha='center', fontweight='bold')

# Plot 3: Vocab Compliance at Low ELO
bars3 = axes[2].bar(strat_names, df_summary['Avg Terms at Low ELO'], color=colors)
axes[2].set_title('Avg Advanced Terms at Low ELO\n(Lower = better vocabulary control)', fontweight='bold')
axes[2].set_ylabel('Avg Terms Used')
axes[2].set_xticks(range(len(strat_names)))
axes[2].set_xticklabels(strat_names, rotation=15, ha='right')
for bar, val in zip(bars3, df_summary['Avg Terms at Low ELO']):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{val:.2f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('strategy_calibration_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: strategy_calibration_summary.png")

## Print All Explanations Side-by-Side

In [ ]:
for elo in ELO_LEVELS:
    print(f"\n{'#'*70}")
    print(f"ELO {elo} — {ELO_META[elo]['label']}")
    print(f"{'#'*70}")
    for strategy_name in STRATEGIES:
        print(f"\n--- {strategy_name} ---")
        print(results[strategy_name][elo]['explanation'])

## Final Summary and Key Finding

In [ ]:
print("PROMPT STRATEGY COMPARISON — FINAL SUMMARY")
print("="*70)
print(df_summary.to_string(index=False))
print()

best_monotonicity = df_summary.loc[df_summary['Monotonicity Score'].idxmax(), 'Strategy']
best_range = df_summary.loc[df_summary['FK Range (800→1800)'].idxmax(), 'Strategy']
best_vocab = df_summary.loc[df_summary['Avg Terms at Low ELO'].idxmin(), 'Strategy']

print("KEY FINDINGS:")
print(f"  Best monotonic ELO calibration:  {best_monotonicity}")
print(f"  Widest complexity spread:        {best_range}")
print(f"  Best low-ELO vocabulary control: {best_vocab}")
print()
print("INTERPRETATION:")
print("  Monotonicity Score = 1.0 means reading complexity increased at every ELO step.")
print("  FK Range measures how much the system differentiates language between ELO 800 and 1800.")
print("  Avg Terms at Low ELO measures how well forbidden vocabulary is suppressed for beginners.")
print()
print("  A well-calibrated system scores high on monotonicity and FK range,")
print("  and low on advanced term usage at low ELO.")